# Kumo Relational on AdventureWorks, from Snowflake

The Snowflake peer of `nemotron_relational_adventureworks_databricks.ipynb`. Same
data, same questions. What differs is where the tables live: the graph is
read from a Snowflake schema, and only the prediction leaves the session.

**Runs inside a Snowflake notebook.** It takes the active Snowpark session
with `get_active_session()`, so it cannot be executed from a local
interpreter unchanged. To run it elsewhere, replace that call with a
`snowflake.connector` connection.

**Before you start**, seed the tables with
`seed_adventureworks_snowflake.py`, and set `KUMO_RELATIONAL_API_ENDPOINT` (and
`KUMO_RELATIONAL_API_KEY` if the endpoint is behind a gateway) to a running
NemotronRelational NIM.


In [ ]:
from snowflake.snowpark.context import get_active_session

get_active_session().file.get('@KUMO_RFM_SPCS.SHARED.WHEELS', '/tmp/wheels')

!pip install --no-index --find-links /tmp/wheels \
  'kumo-relational-client[nemotron_relational,snowflake]' 'numpy<2'

## 1. Connect

The notebook already holds a session, so nothing here handles credentials. The same
session serves the graph, the plain SQL further down, and the model call.


In [ ]:
from kumo_relational_client import RelationalClient
from kumo_relational_client.relational import Graph
from snowflake.snowpark.context import get_active_session

DATABASE = 'KUMO_RFM_SPCS'
SCHEMA = 'ADVENTUREWORKS'
SERVICE = 'KUMO_RFM_SPCS.SERVING.KUMO_RFM_SVC'
OUTPUT_TABLE = 'AW_PREDICTIONS'

session = get_active_session()


def query(sql):
    return session.sql(sql).collect()

### Two expected warnings

Both are correct and neither is worth acting on, so they are silenced by exact
text. Anything else the parser or the sampler reports still surfaces.

**Seeded sampling.** Snowflake supports `SEED` only with `SYSTEM` sampling, which
draws blocks rather than a fixed number of rows, so the connector declines it
rather than bias the in-context examples. The consequence is real: `random_seed`
has no effect here and **repeated runs return different predictions**. The
Databricks and DuckDB backends do seed, via `REPEATABLE`, so only Snowflake
varies this way.

**Semantic type.** `SUM` over a `categorical` column is flagged as a mismatch.
That is the deliberate choice described above: `OrderQty` is genuinely
low-cardinality, and declaring it `numerical` makes the forecast measurably
worse.


In [ ]:
import warnings

warnings.filterwarnings(
    'ignore',
    message=r"The 'snowflake' backend does not support seeded random sampling",
)
warnings.filterwarnings(
    'ignore',
    message=(
        r'Encountered the following warnings during parsing:[\s\S]*'
        r'OrderQty has semantic type categorical'
    ),
)

## 2. Build the graph

Primary keys are declared rather than inferred. `SalesOrderID` is unique on the
header, but so is `rowguid`, and inference declines to choose between them, which
leaves the header unlinkable and the detail table with nothing to aggregate through.

`from_snowflake` takes a `SnowflakeConnection`, so it is handed
`session.connection` rather than the Snowpark session itself. The serving client
below takes the session, which is what it uses to issue the model call.


In [ ]:
graph = Graph.from_snowflake(
    connection=session.connection,
    database=DATABASE,
    schema=SCHEMA,
    tables=[
        {
            'name': 'CUSTOMERS',
            'source_name': 'AW_CUSTOMERS',
            'primary_key': 'CUSTOMERID',
        },
        {
            'name': 'PRODUCTS',
            'source_name': 'AW_PRODUCTS',
            'primary_key': 'PRODUCTID',
        },
        {
            'name': 'SALES_ORDER_HEADERS',
            'source_name': 'AW_SALES_ORDER_HEADERS',
            'primary_key': 'SALESORDERID',
        },
        {
            'name': 'SALES_ORDER_DETAILS',
            'source_name': 'AW_SALES_ORDER_DETAILS',
            'primary_key': 'SALESORDERDETAILID',
        },
    ],
)

## 3. Say which tables are timelines

`OrderQty` is left as inferred, which is `categorical`. A column counts as
categorical when its distinct values are few relative to its rows, and an order
line is almost always for one or two items. Declaring it `numerical` looks
obviously right and makes the forecast measurably worse.

`rowguid` is a per-row UUID AdventureWorks carries for replication. It identifies
a row and says nothing about it. Left on `sales_order_details`, the table the
demand query aggregates over, it drives the regression to exactly 0.00 at every
anchor.


In [ ]:
graph['SALES_ORDER_HEADERS'].time_column = 'ORDERDATE'
graph['SALES_ORDER_DETAILS'].time_column = 'ORDERDATE'
graph['CUSTOMERS'].time_column = None
graph['PRODUCTS'].time_column = None

for table_name in (
    'CUSTOMERS',
    'PRODUCTS',
    'SALES_ORDER_HEADERS',
    'SALES_ORDER_DETAILS',
):
    graph[table_name].remove_column('ROWGUID')

graph.print_metadata()
graph.print_links()
graph.validate()

## 4. Point at the NIM

The graph is built and sampled here, in the notebook; only the
prediction itself is sent. `RelationalClient` takes the URL of a running
NemotronRelational NIM, and an `api_key` when one sits behind an
authenticating gateway.


In [ ]:
import os

client = RelationalClient(
    os.environ['KUMO_RELATIONAL_API_ENDPOINT'],
    api_key=os.environ.get('KUMO_RELATIONAL_API_KEY'),
)
model = client.relational(graph)

## 5. Forecast 30-day demand


In [ ]:
import pandas as pd

results = {}

results['demand_30d'] = model.predict(
    'PREDICT SUM(SALES_ORDER_DETAILS.ORDERQTY, 0, 30, days) '
    'FOR EACH PRODUCTS.PRODUCTID',
    indices=[707, 708, 711],
)
results['demand_30d']

### The same forecast, as of a past date

An anchor time in the past is how a forecast is checked against what actually
happened, rather than waiting 30 days to find out.


In [ ]:
latest = pd.Timestamp(
    query(
        f'SELECT MAX(ORDERDATE) AS d FROM {DATABASE}.{SCHEMA}.AW_SALES_ORDER_DETAILS'
    )[0]['D']
)
anchor = latest - pd.Timedelta(days=30)

results['demand_30d_past'] = model.predict(
    'PREDICT SUM(SALES_ORDER_DETAILS.ORDERQTY, 0, 30, days) '
    'FOR EACH PRODUCTS.PRODUCTID',
    indices=[707, 708, 711],
    anchor_time=anchor,
)
results['demand_30d_past']

### Several timeframes at once

Forecasting takes one entity at a time: the three rows come back as three
timeframes for the same product, not one row per product.


In [ ]:
results['demand_3_timeframes'] = model.predict(
    'PREDICT SUM(SALES_ORDER_DETAILS.ORDERQTY, 0, 30, days) '
    'FORECAST 3 TIMEFRAMES FOR EACH PRODUCTS.PRODUCTID',
    indices=[707],
)
results['demand_3_timeframes']

## 6. Who is about to go quiet, and what to offer them


In [ ]:
recent_customers = [
    row['CUSTOMERID']
    for row in query(f"""
    SELECT DISTINCT CUSTOMERID AS CUSTOMERID
    FROM {DATABASE}.{SCHEMA}.AW_SALES_ORDER_HEADERS
    ORDER BY CUSTOMERID
    LIMIT 100
""")
]

results['churn_30d'] = model.predict(
    'PREDICT COUNT(SALES_ORDER_HEADERS.*, 0, 30, days) = 0 '
    'FOR EACH CUSTOMERS.CUSTOMERID',
    indices=recent_customers,
)
results['churn_30d'].head()

Recommendation ranks products a customer has not bought yet. This is a
link-prediction query: at most 200 entities per request.


In [ ]:
results['recommend_top10'] = model.predict(
    'PREDICT LIST_DISTINCT(SALES_ORDER_DETAILS.PRODUCTID, 0, 30, days) '
    'RANK TOP 10 FOR EACH CUSTOMERS.CUSTOMERID',
    indices=recent_customers,
)
results['recommend_top10'].head()

## 7. Fill in a missing attribute

About half the products carry no colour. This asks the model for the missing value
rather than a future event, which is the same machinery pointed at a static column.


In [ ]:
results['product_color'] = model.predict(
    'PREDICT PRODUCTS.COLOR FOR EACH PRODUCTS.PRODUCTID',
    indices=[707, 708, 711, 712, 713],
)
results['product_color']

## 8. Save the predictions


In [ ]:
frames = []
for name, frame in results.items():
    tagged = frame.copy()
    tagged.insert(0, 'QUERY_NAME', name)
    frames.append(tagged.astype(dict.fromkeys(tagged.columns, str)))

combined = pd.concat(frames, ignore_index=True)
combined['PREDICTED_AT'] = pd.Timestamp.utcnow().strftime('%Y-%m-%d %H:%M:%S')

session.write_pandas(
    combined,
    OUTPUT_TABLE,
    database=DATABASE,
    schema=SCHEMA,
    auto_create_table=True,
    overwrite=True,
    quote_identifiers=False,
)
query(f'SELECT * FROM {DATABASE}.{SCHEMA}.{OUTPUT_TABLE} LIMIT 10')

## 9. Close the client

Nothing is torn down here. The compute pool suspends itself after 30 minutes
idle and resumes on the next call, so the service is left for the next reader
rather than stopped from inside a notebook that does not own it.


In [ ]:
client.close()